# Final Project: ปัจจัยที่เกี่ยวข้องกับการเข้ารับการรักษาสุขภาพจิตของคนทำงานสาย Tech
> 1145 201 คณิตศาสตร์สำหรับวิทยาการข้อมูล | Semester 1/2568

**ผู้จัดทำ (ทำคนเดียว):** [ชื่อ-นามสกุล] ([รหัสนักศึกษา])

**Dataset:** OSMI Mental Health in Tech Survey 2014 (แบบสำรวจของ Open Sourcing Mental Illness, เผยแพร่บน Kaggle) ไฟล์ `dataset.csv` — 1,259 แถว, 27 คอลัมน์

**Problem Statement:** แบบสำรวจนี้ถามพนักงานในอุตสาหกรรม Tech ว่าเคยเข้ารับการรักษาสุขภาพจิตหรือไม่ (`treatment`) ร่วมกับข้อมูลส่วนตัว นโยบายของบริษัท และทัศนคติของที่ทำงาน โครงงานนี้ต้องการตอบว่า **ปัจจัยใดสัมพันธ์กับการเข้ารับการรักษามากที่สุด และทำนาย `treatment` (Yes/No) ได้แม่นยำแค่ไหน** ซึ่งเป็นปัญหา **Classification** ที่ target สมดุลดี (Yes 50.6% / No 49.4%)

---

| CLO | ส่วนที่ครอบคลุม | Status |
|-----|----------------|--------|
| CLO1 (Linear Algebra) | Part 2 — PCA ด้วย eigendecomposition + ตรวจกับ SVD | ✅ |
| CLO2 (Statistical Learning) | Part 3 — Chi-square, t-test, Bias-Variance U-curve | ✅ |
| CLO3 (Regression/Classification) | Part 4 — Logistic Regression, KNN | ✅ |
| CLO4 (Model Selection) | Part 5 — 5-Fold CV, 4 models | ✅ |

In [ ]:
# ─── Import libraries ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
print('Ready.')

---
## Part 1: Dataset Overview และ EDA เบื้องต้น

โหลดข้อมูล ตรวจคุณภาพ ทำความสะอาด แล้วทำ EDA เบื้องต้น

In [ ]:
# ─── โหลด Dataset ────────────────────────────────────────────────
# วัตถุประสงค์: เข้าใจโครงสร้างของข้อมูลก่อนวิเคราะห์
df_raw = pd.read_csv('dataset.csv')   # OSMI Mental Health in Tech Survey 2014

print(f'Shape: {df_raw.shape}')
print(f'\nMissing values (เฉพาะคอลัมน์ที่มี missing):')
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])
print(f'\nTarget (treatment) distribution:')
print(df_raw['treatment'].value_counts())
df_raw.head()

**ปัญหาคุณภาพข้อมูลที่พบและวิธีจัดการ**

| ปัญหา | วิธีจัดการ |
|---|---|
| `Age` มีค่าเป็นไปไม่ได้ (เช่น -1726, 99999999999) | ค่านอกช่วง 18–75 ตั้งเป็น NaN แล้วแทนด้วย median |
| `Gender` พิมพ์ไม่เหมือนกัน (Male/male/M/Make/...) 49 แบบ | รวมเป็น Male / Female / Other |
| `Country` มี 48 ประเทศ | เก็บ 4 ประเทศหลัก ที่เหลือเป็น Other |
| `state` missing 41% และใช้ได้เฉพาะสหรัฐฯ | ตัดทิ้ง |
| `comments` missing 87%, `Timestamp` ไม่ใช่ feature | ตัดทิ้ง |
| `self_employed` missing 18 แถว | เติมด้วย mode |
| `work_interfere` missing 264 แถว และเป็น leakage | ตัดทิ้ง (อธิบายด้านล่าง) |

In [ ]:
# ─── Data Cleaning ────────────────────────────────────────────────
# วัตถุประสงค์: จัดการ Age ผิดปกติ, Gender ที่พิมพ์หลายแบบ, missing values
df = df_raw.copy()

# 1) Age: ค่าที่เป็นไปไม่ได้ (เช่น -1726, 99999999999) ให้เป็น NaN แล้วแทนด้วย median
n_bad_age = ((df['Age'] < 18) | (df['Age'] > 75)).sum()
df.loc[(df['Age'] < 18) | (df['Age'] > 75), 'Age'] = np.nan
df['Age'] = df['Age'].fillna(df['Age'].median())
print(f'แก้ Age ผิดปกติ {n_bad_age} แถว')

# 2) Gender: รวมคำที่พิมพ์ต่างกันให้เหลือ 3 กลุ่ม Male / Female / Other
def clean_gender(g):
    g = str(g).strip().lower()
    male = ['male', 'm', 'man', 'make', 'male ', 'cis male', 'cis man', 'male (cis)',
            'mal', 'maile', 'msle', 'mail', 'malr', 'guy (-ish) ^_^']
    female = ['female', 'f', 'woman', 'femake', 'female ', 'cis female', 'female (cis)',
              'femail', 'cis-female/femme']
    if g in male: return 'Male'
    if g in female: return 'Female'
    return 'Other'
df['Gender'] = df['Gender'].apply(clean_gender)
print(df['Gender'].value_counts())

# 3) Country: เก็บ 4 ประเทศที่พบมากสุด ที่เหลือรวมเป็น Other
top_countries = df['Country'].value_counts().nlargest(4).index
df['Country'] = df['Country'].where(df['Country'].isin(top_countries), 'Other')

# 4) self_employed: เติมด้วย mode (missing เพียง 18 แถว)
df['self_employed'] = df['self_employed'].fillna(df['self_employed'].mode()[0])

# 5) ตัดคอลัมน์ที่ไม่ใช้
#    - Timestamp, comments: ไม่ใช่ feature เชิงวิเคราะห์ (comments ว่าง 87%)
#    - state: ใช้ได้เฉพาะสหรัฐฯ และ missing 41%
#    - work_interfere: ตอบเฉพาะผู้ที่มีภาวะอยู่แล้ว (NA = ไม่มี) จึงเป็น "target leakage"
#      เกือบตรงๆ กับ treatment -> ไม่นำมาใช้ทำนาย (ดูเหตุผลในเซลล์ถัดไป)
df = df.drop(columns=['Timestamp', 'comments', 'state', 'work_interfere'])

# 6) Target: treatment Yes=1, No=0
df['treatment'] = (df['treatment'] == 'Yes').astype(int)

print(f'\nShape หลัง clean: {df.shape}, missing ทั้งหมด = {df.isnull().sum().sum()}')

In [ ]:
# ─── เหตุผลที่ตัด work_interfere (Target Leakage) ──────────────────
# ดู crosstab กับ treatment
ct = pd.crosstab(df_raw['work_interfere'].fillna('NA (ไม่มีภาวะ)'), df_raw['treatment'])
print(ct)
# ผู้ตอบ NA ส่วนใหญ่ treatment=No, ส่วน Often/Sometimes ส่วนใหญ่ treatment=Yes
# -> ตัวแปรนี้ "รู้คำตอบ" ของ target อยู่แล้ว ถ้าใส่เข้าไป accuracy จะสูงเกินจริง

**ทำไมต้องตัด `work_interfere`:** คำถามนี้ถามว่า "ถ้ามีภาวะทางสุขภาพจิต อาการรบกวนงานบ่อยแค่ไหน" ผู้ที่ไม่มีภาวะจะไม่ตอบ (NA) จากตาราง ผู้ตอบ NA มี treatment=No ถึง 260 จาก 264 คน ขณะที่ผู้ตอบ Often มี treatment=Yes 123 จาก 144 คน ตัวแปรนี้จึงบอกคำตอบของ target เกือบตรงๆ ถ้าใส่เข้าไปจะได้ accuracy ที่ดูดีแต่ไม่สะท้อนความสามารถทำนายจริง จึงตัดออกเพื่อให้การประเมินโมเดลสมเหตุสมผล

In [ ]:
# ─── Descriptive Statistics + Visualization ───────────────────────
print(df[['Age']].describe().round(2))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# (1) Distribution ของ target
df['treatment'].map({0: 'No', 1: 'Yes'}).value_counts().plot(kind='bar', ax=axes[0], color=['#4C72B0', '#DD8452'], rot=0)
axes[0].set_title('Target: treatment'); axes[0].set_ylabel('จำนวน')

# (2) Age แยกตาม treatment
for t, c in zip([0, 1], ['#4C72B0', '#DD8452']):
    axes[1].hist(df.loc[df.treatment == t, 'Age'], bins=20, alpha=0.6, color=c, label=f'treatment={t}')
axes[1].set_title('Age แยกตาม treatment'); axes[1].legend()

# (3) สัดส่วน treatment ตาม family_history
df.groupby('family_history')['treatment'].mean().plot(kind='bar', ax=axes[2], color='#55A868', rot=0)
axes[2].set_title('สัดส่วนที่รับการรักษา ตาม family_history'); axes[2].set_ylabel('P(treatment=1)')
plt.tight_layout(); plt.show()

In [ ]:
# ─── เตรียม Feature Matrix ────────────────────────────────────────
# One-hot encoding ตัวแปรเชิงกลุ่ม (drop_first=True เพื่อเลี่ยง multicollinearity)
y = df['treatment'].values
X_df = pd.get_dummies(df.drop(columns='treatment'), drop_first=True).astype(float)
feature_names = X_df.columns.tolist()
X = X_df.values
print(f'X shape = {X.shape}  (แถว x features หลัง one-hot)')

# Heatmap: correlation ของ features กับ target (top 15 ตามค่าสัมบูรณ์)
corr_target = X_df.corrwith(pd.Series(y, index=X_df.index)).sort_values(key=abs, ascending=False)
top15 = corr_target.head(15)
plt.figure(figsize=(7, 5))
sns.barplot(x=top15.values, y=top15.index, palette='RdBu_r', hue=top15.index, legend=False)
plt.title('Top 15 features ที่ correlate กับ treatment'); plt.xlabel('Pearson r'); plt.tight_layout(); plt.show()

**Insight จาก EDA:**
> - Target สมดุลดี (Yes 637 / No 622) จึงใช้ Accuracy เป็น metric หลักได้ และ baseline (ทายคลาสส่วนใหญ่) อยู่ที่ราว 50.6%
> - อายุเฉลี่ยประมาณ 32 ปี (ช่วง 18–72) กลุ่มที่เคยรักษามีอายุสูงกว่าเล็กน้อย
> - `family_history` แยกกลุ่มได้ชัดที่สุด คนที่มีประวัติครอบครัวมีสัดส่วนเข้ารับการรักษาสูงกว่าอย่างเห็นได้ชัด
> - หลัง one-hot encoding ได้ 45 features สำหรับใช้ในส่วนต่อไป

---
## Part 2: CLO1 — Linear Algebra Analysis

**เลือกทำ Task A (PCA) และ Task C (Eigendecomposition ของ covariance matrix)** โดยเขียน PCA เองด้วย `np.linalg.eigh` บน covariance matrix ของข้อมูลที่ standardize แล้ว จากนั้นตรวจความถูกต้องเทียบกับ `sklearn.PCA` และ SVD (ความสัมพันธ์ $\lambda_i = \sigma_i^2/(n-1)$)

**คำอธิบาย:** survey มี 45 features ที่มักตอบสัมพันธ์กัน (เช่น บริษัทที่มี benefits มักมี wellness program) จึงใช้ PCA เพื่อดูว่าโครงสร้างของคำตอบมีกี่มิติหลัก และ PC แรกๆ สะท้อนอะไร

In [ ]:
# ─── CLO1: PCA ด้วย Eigendecomposition (เขียนเองด้วย NumPy) ─────────
# วัตถุประสงค์: ลด dimension และดูว่าโครงสร้างความแปรปรวนของ features อยู่ที่ไหน

# 1) Standardize แล้วสร้าง covariance matrix  C = (1/(n-1)) Xc^T Xc
scaler_pca = StandardScaler()
X_std = scaler_pca.fit_transform(X)
n = X_std.shape[0]
C_mat = (X_std.T @ X_std) / (n - 1)

# 2) Eigendecomposition (eigh เหมาะกับ symmetric matrix) แล้วเรียงจากมากไปน้อย
eigvals, eigvecs = np.linalg.eigh(C_mat)
idx = np.argsort(eigvals)[::-1]
eigvals, eigvecs = eigvals[idx], eigvecs[:, idx]

# 3) Explained variance ratio = lambda_i / sum(lambda)
evr = eigvals / eigvals.sum()
cum_evr = np.cumsum(evr)
k90 = int(np.argmax(cum_evr >= 0.90)) + 1
print(f'PC1 = {evr[0]:.1%}, PC2 = {evr[1]:.1%}, PC3 = {evr[2]:.1%}')
print(f'ต้องใช้ {k90} PCs จากทั้งหมด {len(evr)} เพื่ออธิบายความแปรปรวน >= 90%')

# 4) ตรวจสอบความถูกต้อง: เทียบกับ sklearn PCA และ SVD
pca_sk = PCA().fit(X_std)
U, S, Vt = np.linalg.svd(X_std, full_matrices=False)
print('eigendecomp == sklearn PCA :', np.allclose(evr, pca_sk.explained_variance_ratio_))
print('eigenvalue == sigma^2/(n-1) (SVD):', np.allclose(eigvals, S**2 / (n - 1)))

# 5) Scree plot + cumulative variance
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].plot(range(1, 21), evr[:20], 'o-')
axes[0].set_title('Scree Plot (20 PCs แรก)'); axes[0].set_xlabel('PC'); axes[0].set_ylabel('Explained variance ratio')
axes[1].plot(range(1, len(cum_evr) + 1), cum_evr, '-')
axes[1].axhline(0.9, color='r', ls='--', label='90%'); axes[1].axvline(k90, color='gray', ls=':')
axes[1].set_title('Cumulative Explained Variance'); axes[1].set_xlabel('จำนวน PCs'); axes[1].legend()

# 6) Project ลง 2D แล้วสีตาม treatment
Z = X_std @ eigvecs[:, :2]
sc = axes[2].scatter(Z[:, 0], Z[:, 1], c=y, cmap='coolwarm', alpha=0.5, s=12)
axes[2].set_title('2D PCA projection (สี = treatment)'); axes[2].set_xlabel('PC1'); axes[2].set_ylabel('PC2')
plt.colorbar(sc, ax=axes[2], label='treatment')
plt.tight_layout(); plt.show()

# 7) Loadings: features ที่ contribute มากสุดใน PC1 และ PC2
for k in range(2):
    load = pd.Series(eigvecs[:, k], index=feature_names)
    print(f'\nTop loadings PC{k+1}:')
    print(load.reindex(load.abs().sort_values(ascending=False).index).head(6).round(3))

In [ ]:
# ─── PCA ช่วยแยก treatment ได้แค่ไหน? ───────────────────────────────
# เทียบค่าเฉลี่ยของ PC1, PC2 ระหว่างสองกลุ่ม + t-test
for k in range(2):
    g1, g0 = Z[y == 1, k], Z[y == 0, k]
    t, p = stats.ttest_ind(g1, g0, equal_var=False)
    print(f'PC{k+1}: mean(treat=1)={g1.mean():.3f}, mean(treat=0)={g0.mean():.3f}, t={t:.2f}, p={p:.4f}')

**Insight จาก CLO1 Analysis:**
> - **PC1 อธิบายความแปรปรวน 10.1%, PC2 8.7%, PC3 5.7%** และต้องใช้ถึง **32 จาก 45 PCs** เพื่อถึง 90% แสดงว่าข้อมูลไม่ได้อยู่ในมิติต่ำ เพราะ features เป็นตัวแปรเชิงกลุ่มที่ one-hot แล้ว ความสัมพันธ์ระหว่างกันจึงกระจายหลายทิศทาง
> - **PC1 คือแกน "การสนับสนุนจากนายจ้าง"**: loadings สูงสุดคือ `wellness_program`, `seek_help`, `benefits` โดย Yes กับ No มีเครื่องหมายตรงข้ามกัน บริษัทที่มีสวัสดิการด้านสุขภาพจิตจะอยู่ปลายด้านหนึ่ง
> - **PC2 คือแกน "ความกล้าเปิดเผย/ผลกระทบที่คาดไว้"**: loadings สูงสุดคือ `mental_health_consequence`, `supervisor`, `coworkers`, `mental_health_interview`
> - ในแผนภาพ 2D สองกลุ่ม treatment **ซ้อนทับกันมาก** แต่ค่าเฉลี่ยบน PC1 และ PC2 ต่างกันอย่างมีนัยสำคัญ (Welch t-test: PC1 p=0.0076, PC2 p<0.0001) แปลว่า PCA แบบไม่ใช้ label จับสัญญาณได้บางส่วน แต่ไม่พอแยกกลุ่มได้เอง จึงต้องใช้โมเดลใน Part 4
> - eigendecomposition ที่เขียนเองให้ผลตรงกับ sklearn PCA และ SVD (`True` ทั้งสองการตรวจ)

---
## Part 3: CLO2 — Statistical Learning

**สิ่งที่ทำ:** distribution ของ features สำคัญ, ทดสอบทางสถิติระหว่าง features กับ target (Chi-square, Welch t-test), และ Bias-Variance U-curve ด้วย KNN

**คำถาม: Dataset นี้เหมาะกับ Regression หรือ Classification?**
> **Classification** เพราะ target `treatment` เป็นตัวแปรสองค่า (Yes/No) ไม่ใช่ตัวเลขต่อเนื่อง

In [ ]:
# ─── CLO2: Distribution + Correlation กับ Target ──────────────────
cat_cols = ['family_history', 'benefits', 'care_options', 'anonymity', 'mental_health_consequence']
fig, axes = plt.subplots(1, len(cat_cols), figsize=(18, 3.8), sharey=True)
for ax, col in zip(axes, cat_cols):
    df.groupby(col)['treatment'].mean().sort_values().plot(kind='barh', ax=ax, color='#4C72B0')
    ax.set_title(col); ax.set_xlabel('P(treatment=1)')
plt.tight_layout(); plt.show()

In [ ]:
# ─── Statistical Tests ────────────────────────────────────────────
# (1) Chi-square test of independence: ตัวแปรเชิงกลุ่ม vs treatment
rows = []
for col in df.select_dtypes(include='object').columns:
    table = pd.crosstab(df[col], df['treatment'])
    chi2, p, dof, _ = stats.chi2_contingency(table)
    cramers_v = np.sqrt(chi2 / (len(df) * (min(table.shape) - 1)))
    rows.append((col, chi2, dof, p, cramers_v))
chi_df = pd.DataFrame(rows, columns=['feature', 'chi2', 'dof', 'p_value', 'cramers_V']).sort_values('p_value')
print(chi_df.round(4).head(10).to_string(index=False))

# (2) Welch t-test: Age ระหว่างกลุ่มที่รักษา vs ไม่รักษา
t, p = stats.ttest_ind(df.loc[df.treatment == 1, 'Age'], df.loc[df.treatment == 0, 'Age'], equal_var=False)
print(f'\nWelch t-test Age: mean(treat=1)={df.loc[df.treatment==1,"Age"].mean():.2f}, '
      f'mean(treat=0)={df.loc[df.treatment==0,"Age"].mean():.2f}, t={t:.2f}, p={p:.4f}')

In [ ]:
# ─── Bias-Variance: Train/Test error vs Model Complexity ──────────
# ใช้ KNN: k เล็ก = โมเดลซับซ้อน (variance สูง), k ใหญ่ = เรียบ (bias สูง)
# แบ่ง train/test (stratify เพื่อรักษาสัดส่วนคลาส) ก่อน scale เพื่อป้องกัน data leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_train)          # fit เฉพาะ train
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

ks = list(range(1, 151, 2))
train_err, test_err = [], []
for k in ks:
    knn = KNeighborsClassifier(n_neighbors=k).fit(X_train_s, y_train)
    train_err.append(1 - knn.score(X_train_s, y_train))
    test_err.append(1 - knn.score(X_test_s, y_test))

best_k_bv = ks[int(np.argmin(test_err))]
plt.figure(figsize=(8, 4.5))
plt.plot(1 / np.array(ks), train_err, 'o-', ms=3, label='Train error')
plt.plot(1 / np.array(ks), test_err, 's-', ms=3, label='Test error')
plt.xlabel('Model complexity (1/k)  →  ซับซ้อนขึ้น'); plt.ylabel('Misclassification error')
plt.title('Bias-Variance U-curve (KNN)'); plt.legend(); plt.tight_layout(); plt.show()
print(f'k ที่ test error ต่ำสุด = {best_k_bv} (test error = {min(test_err):.3f})')
print(f'k=1: train error = {train_err[0]:.3f}, test error = {test_err[0]:.3f}')

**Insight จาก CLO2 Analysis:**
> - **Features ที่สัมพันธ์กับ treatment มากที่สุด:** `family_history` (Cramér's V = 0.376, χ²=178.3, p<0.001), ตามด้วย `care_options` (V=0.274), `benefits` (V=0.227) และ `Gender` (V=0.202) ทุกตัวมี p<0.001 ส่วน `Age` ต่างกันอย่างมีนัยสำคัญ (32.6 vs 31.5 ปี, p=0.009) แต่ต่างกันเพียง ~1 ปี effect เล็ก
> - **ข้อควรระวัง:** นี่คือความสัมพันธ์ ไม่ใช่เหตุและผล และข้อมูลมาจากการสมัครใจตอบแบบสำรวจ (self-selection)
> - **Bias-Variance:** ที่ k=1 (ซับซ้อนสุด) train error = 0 แต่ test error = 0.409 ซึ่ง overfit ชัดเจน (variance สูง) เมื่อเพิ่ม k test error ลดลงจนต่ำสุดที่ **k≈45 (test error 0.310)** แล้วเริ่มเพิ่มขึ้นอีกเมื่อโมเดลเรียบเกินไป (bias สูง) ได้กราฟรูปตัว U ตามทฤษฎี
> - **Optimal complexity** อยู่ที่ k ราว 45–50 ซึ่งสอดคล้องกับ k=51 ที่เลือกด้วย CV ใน Part 4

---
## Part 4: CLO3 — Model Building

สร้าง Classification model 2 ตัว: **Logistic Regression** และ **KNN** โดยแบ่ง train/test 80/20 (stratify) ก่อน แล้ว standardize โดย fit scaler เฉพาะ train เท่านั้น

**Dataset ของกลุ่มเหมาะกับ:** ⬜ Regression  ✅ **Classification**

In [ ]:
# ─── CLO3: Model 1 — Logistic Regression ──────────────────────────
# ใช้ X_train_s / X_test_s ที่ standardize แล้วจากเซลล์ Bias-Variance
model1 = LogisticRegression(max_iter=1000)
model1.fit(X_train_s, y_train)

pred1 = model1.predict(X_test_s)
train_acc1 = model1.score(X_train_s, y_train)
test_acc1 = accuracy_score(y_test, pred1)
print(f'Logistic Regression: Train Acc = {train_acc1:.3f}, Test Acc = {test_acc1:.3f}')
print(classification_report(y_test, pred1, target_names=['No', 'Yes']))

# Coefficients (หลัง standardize เทียบขนาดกันได้) -> เป็น log-odds ต่อ 1 SD
coef = pd.Series(model1.coef_[0], index=feature_names).sort_values(key=abs, ascending=False)
print('Top 8 coefficients:'); print(coef.head(8).round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred1), display_labels=['No', 'Yes']).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix: Logistic Regression')
coef.head(10)[::-1].plot(kind='barh', ax=axes[1], color='#4C72B0')
axes[1].set_title('Top 10 coefficients (log-odds ต่อ 1 SD)')
plt.tight_layout(); plt.show()

In [ ]:
# ─── CLO3: Model 2 — KNN Classifier ──────────────────────────────
# เลือก k ด้วย CV บน "train set เท่านั้น" (ไม่แตะ test set)
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
k_grid = [3, 5, 7, 11, 15, 21, 31, 51, 71, 101]
cv_acc_k = [cross_val_score(KNeighborsClassifier(n_neighbors=k), X_train_s, y_train, cv=cv5).mean() for k in k_grid]
best_k = k_grid[int(np.argmax(cv_acc_k))]
print(f'k ที่ดีที่สุดจาก CV (บน train) = {best_k}, CV acc = {max(cv_acc_k):.3f}')

model2 = KNeighborsClassifier(n_neighbors=best_k).fit(X_train_s, y_train)
pred2 = model2.predict(X_test_s)
train_acc2 = model2.score(X_train_s, y_train)
test_acc2 = accuracy_score(y_test, pred2)
print(f'KNN (k={best_k}): Train Acc = {train_acc2:.3f}, Test Acc = {test_acc2:.3f}')
print(classification_report(y_test, pred2, target_names=['No', 'Yes']))

ConfusionMatrixDisplay(confusion_matrix(y_test, pred2), display_labels=['No', 'Yes']).plot(colorbar=False)
plt.title(f'Confusion Matrix: KNN (k={best_k})'); plt.show()

In [ ]:
# ─── Bonus: Bootstrap ประมาณความไม่แน่นอนของ Coefficient ──────────
# ทำ resample train set 300 ครั้ง แล้วดู 95% CI ของ coefficient ที่สำคัญ
rng = np.random.default_rng(42)
top_feats = coef.head(6).index.tolist()
top_idx = [feature_names.index(f) for f in top_feats]
boot_coefs = []
for _ in range(300):
    b = rng.integers(0, len(X_train_s), len(X_train_s))
    m = LogisticRegression(max_iter=1000).fit(X_train_s[b], y_train[b])
    boot_coefs.append(m.coef_[0][top_idx])
boot_coefs = np.array(boot_coefs)
ci = pd.DataFrame({'coef': coef[top_feats].values,
                   'CI_low': np.percentile(boot_coefs, 2.5, axis=0),
                   'CI_high': np.percentile(boot_coefs, 97.5, axis=0)}, index=top_feats).round(3)
ci['CI ไม่คร่อม 0'] = (ci.CI_low > 0) | (ci.CI_high < 0)
print(ci)

**สรุป Model Comparison:**

| Model | Train Accuracy | Test Accuracy | รายละเอียด |
|-------|-------------|------------|----------|
| Model 1: Logistic Regression | 0.737 | **0.710** | Precision/Recall สมดุลทั้งสองคลาส (F1 ≈ 0.70–0.72) |
| Model 2: KNN (k=51) | 0.686 | 0.687 | Recall ของคลาส Yes ต่ำ (0.50) ทำนาย "No" เกินจริง |

**Insight:**
> - **Logistic Regression ดีกว่า KNN** เล็กน้อย (0.710 vs 0.687) และสมดุลกว่า: KNN จับคนที่รักษาจริงได้เพียง 50% (recall ของ Yes) ขณะที่ Logistic ได้ 68%
> - ช่วง train–test ของทั้งสองโมเดลห่างกันไม่มาก (Logistic 0.737→0.710, KNN 0.686→0.687) **ไม่มี overfitting รุนแรง** โมเดลค่อนข้าง underfit คือ accuracy ราว 70% แสดงว่า features ในแบบสำรวจมีสัญญาณจำกัด (และเราตัด `work_interfere` ที่เป็น leakage ออกแล้ว)
> - **ตีความ coefficients (log-odds ต่อ 1 SD):** `family_history_Yes` (+0.82) เป็นปัจจัยบวกที่แรงที่สุด รองลงมาคือ `coworkers_Yes` (+0.56) และ `care_options_Yes` (+0.43) ส่วน `Gender_Male` (−0.28) เป็นลบ คือผู้ชายมีแนวโน้มรายงานการรักษาต่ำกว่าเมื่อควบคุมปัจจัยอื่น
> - **Bootstrap 95% CI** (300 รอบ) ของ 6 coefficients อันดับต้นไม่คร่อมศูนย์ทั้งหมด แสดงว่าเครื่องหมายและความสำคัญของปัจจัยเหล่านี้ค่อนข้างเสถียร
> - ข้อสังเกต: `coworkers_Yes` เป็นบวกอาจเพราะคนที่เคยรักษาเปิดใจกับเพื่อนร่วมงานมากกว่า (ทิศทางเหตุผลอาจกลับกัน)

---
## Part 5: CLO4 — Model Selection

เปรียบเทียบ **4 models** ด้วย **5-Fold Stratified Cross-Validation** บนข้อมูลทั้งหมด โดยใส่ `StandardScaler` ไว้ใน Pipeline เพื่อให้ scaler ถูก fit ใหม่ในแต่ละ fold (ไม่ให้ข้อมูล validation รั่วเข้ามา)

In [ ]:
# ─── CLO4: 5-Fold Stratified Cross-Validation ────────────────────
# ใช้ Pipeline (scale อยู่ใน pipeline) เพื่อให้ scaler ถูก fit ใหม่ในทุก fold -> ไม่มี leakage
models = {
    'Logistic (C=1)':        make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'Logistic (C=0.01, L2)': make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=1000)),
    f'KNN (k={best_k})':     make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k)),
    'Decision Tree (d=4)':   make_pipeline(StandardScaler(), DecisionTreeClassifier(max_depth=4, random_state=42)),
}

cv_results = {}
for name, mdl in models.items():
    scores = cross_val_score(mdl, X, y, cv=cv5, scoring='accuracy')   # ใช้ข้อมูลทั้งหมด X, y
    cv_results[name] = scores
    print(f'{name:24s}: CV Acc = {scores.mean():.3f} ± {scores.std():.3f}   folds = {np.round(scores, 3)}')

# Bar chart ของ CV Accuracy พร้อม error bar (±1 SD)
names = list(cv_results.keys())
means = [cv_results[n].mean() for n in names]
stds = [cv_results[n].std() for n in names]
plt.figure(figsize=(9, 4.5))
bars = plt.bar(names, means, yerr=stds, capsize=5, color=['#4C72B0', '#55A868', '#DD8452', '#C44E52'])
plt.axhline(max(y.mean(), 1 - y.mean()), color='gray', ls='--', label='Baseline (majority class)')
plt.ylim(0.4, max(means) + 0.1); plt.ylabel('CV Accuracy'); plt.title('5-Fold CV: เปรียบเทียบ 4 models')
plt.xticks(rotation=15); plt.legend()
for b, m in zip(bars, means): plt.text(b.get_x() + b.get_width()/2, m + 0.012, f'{m:.3f}', ha='center')
plt.tight_layout(); plt.show()

best_name = names[int(np.argmax(means))]
print(f'\nBest model จาก CV: {best_name}')

**Best Model จาก Cross-Validation:** Logistic Regression (C=1) — CV Accuracy = **0.712 ± 0.015**

| Model | CV Accuracy (mean ± SD) |
|---|---|
| Logistic (C=1) | 0.712 ± 0.015 |
| Logistic (C=0.01, L2) | 0.709 ± 0.019 |
| Decision Tree (depth=4) | 0.696 ± 0.008 |
| KNN (k=51) | 0.673 ± 0.027 |

**เหตุผล:**
> - Logistic (C=1) ได้ค่าเฉลี่ยสูงสุด และสูงกว่า baseline (50.6%) ประมาณ 20 จุดเปอร์เซ็นต์
> - **ข้อควรระวัง:** Logistic ทั้งสองแบบต่างกันเพียง 0.003 ซึ่งน้อยกว่า SD ของ fold (≈0.015–0.019) จึง **ไม่ถือว่าต่างกันอย่างมีนัยสำคัญ** การเลือก C=1 เป็นเพราะค่าเฉลี่ยสูงสุดและตีความง่าย ไม่ได้แปลว่าชนะขาด
> - KNN ต่ำสุดและ SD สูงสุด (0.027) ผลไม่นิ่ง สอดคล้องกับ Bias-Variance: ข้อมูลเป็น one-hot มิติสูง (45 มิติ) ระยะทางจึงแยกกลุ่มได้ไม่ดี
> - Decision Tree ลึก 4 ชั้นมี SD ต่ำสุด (0.008) เสถียรแต่ยังด้อยกว่า Logistic เล็กน้อย แปลว่าความสัมพันธ์ในข้อมูลส่วนใหญ่เป็นแบบเชิงเส้น
> - CV accuracy (0.712) ใกล้เคียง test accuracy ของ Logistic ใน Part 4 (0.710) ยืนยันว่าผลไม่ได้เกิดจากการแบ่ง train/test โดยบังเอิญ

---
## Part 6: สรุปผลและ Data Story

**6.1 สรุปผลการวิเคราะห์:**
- **CLO1:** PCA (เขียนเองด้วย eigendecomposition และตรวจกับ SVD/sklearn) พบว่าคำตอบกระจายหลายมิติ (ต้องใช้ 32/45 PCs สำหรับ 90%) โดย PC1 คือแกนสวัสดิการจากนายจ้าง และ PC2 คือแกนความกล้าเปิดเผย ทั้งสองมีค่าเฉลี่ยต่างกันระหว่างกลุ่ม treatment อย่างมีนัยสำคัญ
- **CLO2:** `family_history` เป็นปัจจัยที่สัมพันธ์กับ treatment แรงที่สุด (V=0.38) ตามด้วย `care_options` และ `benefits`; U-curve ของ KNN ชี้ k≈45 เป็นความซับซ้อนที่เหมาะสม
- **CLO3:** Logistic Regression ได้ Test Accuracy 0.710 ดีกว่า KNN (0.687) และสมดุลกว่า; bootstrap ยืนยันว่า coefficients หลักเสถียร
- **CLO4:** 5-Fold CV เลือก Logistic Regression (0.712 ± 0.015) เป็นโมเดลที่ดีที่สุด แม้ต่างจาก Logistic ที่ regularize มากกว่าอย่างไม่มีนัยสำคัญ

**Data Story:** คนทำงานสาย Tech ที่มี **ประวัติครอบครัว** มีโอกาสเข้ารับการรักษาสูงกว่าอย่างชัดเจน และ **การรู้ว่าบริษัทมีตัวเลือกดูแลสุขภาพจิต** ก็สัมพันธ์กับการรักษาเช่นกัน ขณะที่ผู้ชายรายงานการรักษาน้อยกว่า ปัจจัยทั้งหมดรวมกันทำนายได้ราว 71% ซึ่งดีกว่าเดา (50.6%) แต่ยังไม่สูงมาก แปลว่าการตัดสินใจรักษาขึ้นกับปัจจัยส่วนบุคคลที่แบบสำรวจนี้ไม่ได้วัด

**6.2 Limitations:**
- ข้อมูลปี 2014 และมาจากการสมัครใจตอบ (self-selection bias) ผู้ตอบ 60% มาจากสหรัฐฯ และ 79% เป็นเพศชาย
- ความสัมพันธ์ที่พบไม่ใช่ความสัมพันธ์เชิงเหตุผล (เช่น `coworkers_Yes` อาจเป็นผลของการรักษา ไม่ใช่เหตุ)
- ตัด `work_interfere` เพราะ leakage และ `state` เพราะ missing มาก ทำให้เสียข้อมูลบางส่วน
- Gender กลุ่ม Other มีเพียง 21 คน ผลของกลุ่มนี้ไม่น่าเชื่อถือทางสถิติ
- PCA บนตัวแปร one-hot ไม่เหมาะเท่าตัวแปรต่อเนื่อง (ตีความเป็น "แกนของรูปแบบการตอบ")

**6.3 ขั้นตอนต่อไป (Future Work):**
- ลอง Random Forest / Gradient Boosting และ tune hyperparameters ด้วย nested CV
- ใช้ Multiple Correspondence Analysis (MCA) แทน PCA สำหรับข้อมูลเชิงกลุ่ม
- ประเมินด้วย ROC-AUC และปรับ threshold เพื่อเพิ่ม recall ของคนที่ต้องการรักษา
- เปรียบเทียบกับข้อมูลรอบสำรวจปีหลังๆ ของ OSMI

**6.4 Connection กับ Topics ในวิชา:**
- **Linear Algebra (Eigen/SVD/PCA):** Part 2 เขียน PCA เองและพิสูจน์ $\lambda_i = \sigma_i^2/(n-1)$ เชื่อมกับ Part 3 ที่ใช้ correlation/covariance
- **Bias-Variance:** U-curve ของ KNN (Part 3) อธิบายผล CV ที่ KNN ต่ำและไม่นิ่ง (Part 5)
- **Classification & Standardization:** Logistic/KNN ต้อง standardize ก่อน (Part 4)
- **Cross-Validation & Pipeline:** ป้องกัน data leakage และเลือกโมเดลอย่างเป็นธรรม (Part 5)
- **Bootstrap:** ประมาณ uncertainty ของ coefficient (Part 4, bonus)

---
**Acknowledgements:** ข้อมูลจาก OSMI Mental Health in Tech Survey 2014 (Open Sourcing Mental Illness, ผ่าน Kaggle); เครื่องมือ: pandas, NumPy, SciPy, scikit-learn, matplotlib, seaborn